# P2-02. 흉부 X-ray 폐렴 검출 ② YOLO 학습 + 평가

## YOLO가 뭔가요?
**YOLO (You Only Look Once)**: 사진을 **한 번만** 훑어서 "어디에 무엇이 있는지"를 박스로 한꺼번에 내는 검출 모델이에요.

```
입력 사진 → [CNN 백본: 프로젝트 1의 ResNet처럼 특징 추출]
          → [넥: 여러 해상도의 특징 지도를 합침 (작은 것·큰 것 둘 다 찾으려고)]
          → [헤드: 지도의 각 위치마다 "여기 물체가 있나? 박스는 어디? 확신도는?" 예측]
          → 박스 목록 [(x, y, 가로, 세로, 확신도), ...]
```
- 프로젝트 1의 분류 모델은 마지막에 Global Average Pool로 **위치 정보를 버리고** 라벨 하나만 냈죠. YOLO는 **특징 지도의 위치를 그대로 살려서** "이 위치에 박스"를 내요
- 오늘은 **YOLO26** (Ultralytics 최신 버전, s = small 크기, 약 1천만 파라미터)을 COCO(일상 사진 물체 80종) 사전학습 가중치에서 fine-tuning 해요 → 프로젝트 1의 ImageNet 전이학습과 같은 원리

## 검출 평가 지표 복습
| 지표 | 의미 |
|---|---|
| **IoU** | 예측 박스와 정답 박스가 겹친 넓이 ÷ 합친 넓이 (0~1) |
| **Precision** | 모델이 그린 박스 중 진짜(IoU ≥ 기준)인 비율 |
| **Recall** | 정답 박스 중 모델이 찾아낸 비율 |
| **mAP50** | IoU 0.5 이상을 "맞음"으로 칠 때의 AP (정밀도-재현율 곡선 아래 넓이). **PR-AUC와 같은 개념** |
| **mAP50-95** | IoU 기준을 0.5, 0.55, …, 0.95로 바꿔가며 구한 AP의 평균 → 박스를 **정확하게** 그릴수록 높음 |

> 💡 폐렴 부위는 경계가 흐릿해서 **전문의끼리도 박스가 꽤 다르게** 그려져요. 그래서 이 대회 상위권도 mAP가 그리 높지 않았어요(0.2~0.3대, IoU 0.4~0.75 기준). 프로젝트 1처럼 99%를 기대하면 안 돼요!

**이 노트북에서 답할 질문**
1. YOLO는 폐렴 부위를 얼마나 잘 찾나? (mAP, Precision, Recall)
2. 박스 검출 결과로 **"이 사진에 폐렴이 있나?"(사진 단위 분류)**를 하면 AUROC는?
3. 어떤 사진에서 놓치고(FN), 어떤 사진에 헛박스(FP)를 그리나? **"Not Normal" 음성**에서 오탐이 많을까?
4. **AP vs PA 지름길**: 폐렴이 없는 사진인데 AP라는 이유로 점수가 높게 나오나?

> 실행 후 **2(학습 곡선), 3, 4, 5, 6번 출력**을 Claude에게 보내주세요.

## 0. 준비

In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import font_manager
from PIL import Image
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

_avail = {f.name for f in font_manager.fontManager.ttflist}
for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if _f in _avail:
        plt.rcParams["font.family"] = _f; break
plt.rcParams["axes.unicode_minus"] = False

OUT = Path("../../outputs/p2").resolve()
YOLO_DIR = OUT / "yolo_data"
DATA_YAML = YOLO_DIR / "data.yaml"
meta = pd.read_csv(OUT / "p2_split_meta.csv")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
# Windows + 노트북에서는 데이터 로딩 병렬 처리(workers)가 멈추는 경우가 있어서 0으로 시작해요 (느리지만 안전)
WORKERS = 0 if os.name == "nt" else 4
print("장치:", DEVICE, "| workers:", WORKERS)
print(DATA_YAML.read_text())

## 1. 학습 설정

> 🎯 **각 설정을 왜 이렇게 했나요?**
>
> | 설정 | 값 | 이유 |
> |---|---|---|
> | `model` | `yolo26s.pt` | COCO 사전학습 YOLO26 small. 처음엔 Ultralytics 서버에서 자동 다운로드 (약 20MB) |
> | `imgsz` | 640 | 변환할 때 640으로 저장했고, YOLO 기본 크기 |
> | `epochs` / `patience` | 30 / 8 | 8 epoch 동안 val mAP가 안 오르면 조기 종료 (프로젝트 1과 같은 개념) |
> | `cos_lr` | True | 학습률을 점점 줄이는 cosine (프로젝트 1 노트북 3에서 효과 확인) |
> | `hsv_h`, `hsv_s` | 0 | 색조·채도 증강 끔 → 흑백 X-ray에는 의미 없음 |
> | `hsv_v` | 0.3 | 밝기 증강 → 장비·촬영 조건 차이 흉내 |
> | `fliplr` / `flipud` | 0.5 / 0 | 좌우 반전 OK (폐는 좌우가 비슷), **상하 반전은 해부학적으로 말이 안 돼서 끔** |
> | `degrees` | 5 | ±5° 회전 (촬영 시 몸 기울기 정도) |
> | `mosaic` | 1.0 | 사진 4장을 이어 붙여 1장처럼 학습하는 YOLO 기본 증강. 마지막 10 epoch는 끔(`close_mosaic`) |
>
> ⏱ RTX 3080 기준 epoch당 수 분 → 전체 **1~2시간** 걸릴 수 있어요.  
> 💡 **더 빠르게**: 아래 셀 대신 터미널(`.venv` 켠 상태, 이 노트북 폴더에서)에 출력되는 `yolo` 명령어를 붙여넣으면 workers를 늘려서 돌릴 수 있어요. 끝나면 3번 셀부터 이어서 실행하면 돼요.

In [ ]:
RUN_NAME = "yolo26s"
TRAIN_ARGS = dict(data=str(DATA_YAML), imgsz=640, epochs=30, patience=8, batch=16, device=DEVICE, workers=WORKERS,
                  project=str(OUT / "runs"), name=RUN_NAME, exist_ok=True, seed=42, cos_lr=True,
                  hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, fliplr=0.5, flipud=0.0, degrees=5.0, mosaic=1.0, close_mosaic=10, plots=True)

cli = "yolo detect train model=yolo26s.pt " + " ".join(f"{k}={v}" for k, v in {**TRAIN_ARGS, "workers": 8}.items())
print("터미널로 돌리고 싶다면 이 명령어를 복사하세요:\n")
print(cli)

## 2. 학습 (노트북에서 실행)

In [ ]:
TRAIN_IN_NOTEBOOK = True     # 터미널로 학습했다면 False로 바꾸고 이 셀을 실행 (학습 건너뜀)

RUN_DIR = OUT / "runs" / RUN_NAME
if TRAIN_IN_NOTEBOOK:
    model = YOLO("yolo26s.pt")
    model.train(**TRAIN_ARGS)
BEST = RUN_DIR / "weights" / "best.pt"
print("best 가중치:", BEST, "| 존재:", BEST.exists())

### 2-1. 학습 곡선

> 🎯 **보는 법**: Ultralytics가 자동으로 그려주는 `results.png`예요.  
> - 위쪽 `train/box_loss`(박스 위치 손실), `train/cls_loss`(물체 있나 없나 손실): 내려가야 해요  
> - 아래쪽 `val/...` 손실과 `metrics/mAP50(B)`: val 손실이 다시 올라가거나 mAP가 멈추는 지점이 과적합 시작이에요

In [ ]:
res = pd.read_csv(RUN_DIR / "results.csv"); res.columns = res.columns.str.strip()
best_row = res.loc[res["metrics/mAP50(B)"].idxmax()]
print(f"{len(res)} epoch 학습 | best epoch {int(best_row['epoch'])}: val mAP50 {best_row['metrics/mAP50(B)']:.3f}, "
      f"mAP50-95 {best_row['metrics/mAP50-95(B)']:.3f}, P {best_row['metrics/precision(B)']:.3f}, R {best_row['metrics/recall(B)']:.3f}")
plt.figure(figsize=(16, 8)); plt.imshow(Image.open(RUN_DIR / "results.png")); plt.axis("off"); plt.show()

## 3. Test 평가: 박스 검출 성능 (mAP)

> 🎯 **왜 test에서 따로?** 학습 중 보여준 mAP는 **val** 기준이고, val로 best를 골랐으니 약간 낙관적이에요 (프로젝트 1과 같은 원리). 처음 보는 test로 최종 성적을 재요.  
> 결과 폴더(`runs/test_eval`)에 **PR 곡선, 혼동행렬** 그림도 자동으로 저장돼요.

In [ ]:
best = YOLO(str(BEST))
m = best.val(data=str(DATA_YAML), split="test", imgsz=640, batch=16, device=DEVICE, workers=WORKERS,
             project=str(OUT / "runs"), name="test_eval", exist_ok=True, plots=True, verbose=False)
print(f"\n[TEST] mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f} | Precision {m.box.mp:.3f} | Recall {m.box.mr:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, f in zip(axes, ["BoxPR_curve.png", "BoxF1_curve.png"]):
    p = OUT / "runs" / "test_eval" / f
    if p.exists(): ax.imshow(Image.open(p))
    ax.axis("off")
plt.tight_layout(); plt.show()

## 4. 사진 단위 분류: "이 사진에 폐렴이 있나?"

> 🎯 **왜 하나요?** 임상에서 먼저 궁금한 건 *"박스가 정확한가"*보다 **"이 환자 폐렴 의심인가?"**예요 (응급실 X-ray 우선순위 정하기 등).  
> 검출 결과를 분류로 바꾸는 가장 간단한 방법: **사진의 점수 = 그 사진에서 가장 확신도 높은 박스의 확신도** (박스가 없으면 0)  
> 이 점수로 프로젝트 1에서 익숙한 **AUROC, PR-AUC, 민감도·특이도**를 계산해요.
>
> **기준값(threshold)은 val에서** 정하고 test에 적용해요 (test를 보고 정하면 누수). 여기선 val에서 **민감도 ≥ 0.90**을 만족하는 가장 높은 기준값을 써요 → *폐렴을 놓치지 않는 쪽*을 우선 (선별 검사 관점)

In [ ]:
def predict_split(split):
    files = sorted((YOLO_DIR / "images" / split).glob("*.png"))
    rows, dets = [], {}
    for r in tqdm(best.predict(source=[str(f) for f in files], imgsz=640, conf=0.001, device=DEVICE,
                               stream=True, verbose=False, batch=16), total=len(files), desc=f"{split} 예측"):
        pid = Path(r.path).stem
        b = r.boxes
        dets[pid] = (b.xyxyn.cpu().numpy(), b.conf.cpu().numpy())
        rows.append({"patientId": pid, "score": float(b.conf.max()) if len(b) else 0.0})
    return pd.DataFrame(rows).merge(meta, on="patientId"), dets

val_pred, _ = predict_split("val")
test_pred, test_dets = predict_split("test")

fpr, tpr, thr = roc_curve(val_pred["Target"], val_pred["score"])
THR = float(thr[np.argmax(tpr >= 0.90)])
print(f"val에서 정한 기준값: {THR:.3f} (val 민감도 ≥ 0.90을 만족하는 가장 높은 값)\n")

def summarize(df, thr=THR):
    y, s = df["Target"].values, df["score"].values; p = s >= thr
    tp, fn, fp, tn = (p & (y == 1)).sum(), (~p & (y == 1)).sum(), (p & (y == 0)).sum(), (~p & (y == 0)).sum()
    return {"사진 수": len(df), "폐렴 비율": y.mean(), "AUROC": roc_auc_score(y, s) if 0 < y.mean() < 1 else np.nan,
            "PR-AUC": average_precision_score(y, s) if y.sum() else np.nan,
            "민감도": tp / (tp + fn) if tp + fn else np.nan, "특이도": tn / (tn + fp) if tn + fp else np.nan,
            "PPV": tp / (tp + fp) if tp + fp else np.nan, "NPV": tn / (tn + fn) if tn + fn else np.nan}

display(pd.DataFrame({"val": summarize(val_pred), "test": summarize(test_pred)}).T.round(3))
print("참고: PR-AUC의 '찍기' 기준선은 폐렴 비율(약 0.22)이에요.")

## 5. 하위그룹 분석: AP/PA 지름길 + 어려운 음성 ⭐

> 🎯 **왜 하나요?** P2-01 4번에서 **AP 사진의 폐렴 비율이 PA보다 훨씬 높은** 걸 봤어요. 그럼 모델이 *폐가 아니라 촬영 자세*를 단서로 쓰고 있을지도 몰라요.  
> 확인 방법: **폐렴이 없는(Target=0) 사진만** 모아서 AP와 PA의 점수를 비교해요. 폐렴이 없는데도 **AP라는 이유만으로 점수가 높고 오탐(FP)이 많다면** → 지름길 신호예요.  
> 마찬가지로 음성 중 **"Normal" vs "Not Normal"**의 오탐률을 비교해요. Not Normal은 폐에 *뭔가 이상*이 있는 사진이라 오탐이 많을 거라 예상할 수 있어요.

In [ ]:
rows = []
for (name, col) in [("촬영 자세", "view"), ("성별", "sex"), ("상세 클래스", "class")]:
    for g, d in test_pred.groupby(col):
        rows.append({"구분": name, "그룹": g, **summarize(d)})
sub = pd.DataFrame(rows).set_index(["구분", "그룹"])
display(sub.round(3))

neg = test_pred[test_pred["Target"] == 0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for v, c in [("PA", "#4C72B0"), ("AP", "#C44E52")]:
    axes[0].hist(neg.loc[neg["view"] == v, "score"], bins=40, range=(0, 1), alpha=.6, color=c, label=f"{v} (n={(neg['view'] == v).sum()})", density=True)
axes[0].axvline(THR, ls="--", c="k", lw=1); axes[0].set_title("폐렴 없는 사진의 점수 분포 (AP vs PA)"); axes[0].legend(); axes[0].set_xlabel("사진 점수 (최고 박스 확신도)")
fpr_by = neg.assign(fp=neg["score"] >= THR).groupby(["view", "class"])["fp"].mean().unstack()
(fpr_by * 100).plot.bar(ax=axes[1], rot=0); axes[1].set_title("음성 사진의 오탐률 (%) — 촬영 자세 × 상세 클래스"); axes[1].set_ylabel("%")
plt.tight_layout(); plt.show()
print("오탐률(%) 표:"); display((fpr_by * 100).round(1))

**읽는 법**
- **AP 음성의 오탐률이 PA 음성보다 훨씬 높다** → 모델이 AP 사진 자체를 의심스럽게 봄 (지름길 가능성)
- 단, AP 사진은 실제로 상태가 안 좋은 환자가 많아서 **폐에 다른 이상(Not Normal)이 섞여 있을 가능성**도 커요. 그래서 오른쪽 그래프처럼 **같은 상세 클래스 안에서** AP vs PA를 비교해야 공정해요
  - *Normal 음성끼리 비교해도 AP 오탐이 높다* → 지름길 쪽 증거가 더 강해짐
  - *Not Normal에서만 높다* → 폐의 실제 이상에 반응한 것일 수 있음

## 6. 예측 vs 정답 박스 보기: 맞힌 것, 놓친 것(FN), 헛박스(FP)

> 🎯 숫자 뒤에 있는 실제 사진을 봐요. 빨강 = 전문의 정답 박스, 초록 = 모델 예측 박스(기준값 이상만, 숫자 = 확신도).

In [ ]:
def gt_boxes(pid):
    t = (YOLO_DIR / "labels" / "test" / f"{pid}.txt").read_text().strip()
    return [] if not t else [list(map(float, l.split()[1:])) for l in t.splitlines()]

def show(ax, pid, title):
    im = np.asarray(Image.open(YOLO_DIR / "images" / "test" / f"{pid}.png")); S = im.shape[0]
    ax.imshow(im, cmap="gray"); ax.axis("off"); ax.set_title(title, fontsize=8)
    for cx, cy, w, h in gt_boxes(pid):
        ax.add_patch(patches.Rectangle(((cx - w / 2) * S, (cy - h / 2) * S), w * S, h * S, fill=False, ec="red", lw=2))
    xyxy, conf = test_dets[pid]
    for (x1, y1, x2, y2), c in zip(xyxy, conf):
        if c >= THR:
            ax.add_patch(patches.Rectangle((x1 * S, y1 * S), (x2 - x1) * S, (y2 - y1) * S, fill=False, ec="lime", lw=1.5, ls="--"))
            ax.text(x1 * S, y1 * S - 4, f"{c:.2f}", color="lime", fontsize=7)

tp_ = test_pred[(test_pred["Target"] == 1) & (test_pred["score"] >= THR)].sort_values("score", ascending=False)
fn_ = test_pred[(test_pred["Target"] == 1) & (test_pred["score"] < THR)].sort_values("score")
fp_ = test_pred[(test_pred["Target"] == 0) & (test_pred["score"] >= THR)].sort_values("score", ascending=False)
groups = [("✓ 맞힘 (양성, 높은 점수)", tp_), ("✗ 놓침 FN (양성인데 낮은 점수)", fn_), ("✗ 헛박스 FP (음성인데 높은 점수)", fp_)]
fig, axes = plt.subplots(3, 5, figsize=(16, 10.5))
for r, (name, d) in enumerate(groups):
    for k in range(5):
        if k < len(d):
            row = d.iloc[k]; show(axes[r, k], row["patientId"], f"{name if k == 0 else ''}\n{row['view']} / {row['class'][:20]} / 점수 {row['score']:.2f}")
        else:
            axes[r, k].axis("off")
plt.tight_layout(); plt.show()
print(f"test: 맞힘 {len(tp_)} / 놓침 {len(fn_)} / 헛박스 {len(fp_)} (기준값 {THR:.3f})")

## ✍️ 정리 & 생각해볼 질문

1. test mAP50, mAP50-95는? 프로젝트 1의 99%와 비교해 왜 이렇게 다를까? (힌트: 정답 박스 자체의 애매함)
2. 사진 단위 AUROC는? 박스 성능(mAP)이 낮아도 사진 단위 분류는 잘 될 수 있는 이유는?
3. val 기준 민감도 0.90으로 정한 기준값이 **test에서도 0.90 근처**였나?
4. **AP vs PA**: 음성 사진 오탐률이 AP에서 높았나? *같은 상세 클래스(Normal)* 안에서도 그랬나? → 지름길이라고 말할 수 있나?
5. "Not Normal" 음성의 오탐률은 "Normal"보다 높았나? 그 사진들을 보면 모델이 뭘 폐렴으로 착각한 것 같나?
6. FN(놓친 사진)들은 공통점이 있나? (박스 크기가 작다, 흐릿하다, 심장 뒤에 있다 등)

**메모:**
- 
